# My Search Intelligence Data Contract

[Open in Colab](https://colab.research.google.com/github/vibhanshu-s/FlyrankAI-Internship/blob/main/work/notebooks/w03_data_contract.ipynb)

I am continuing with **Refresh / Content Opportunity Scoring**. This notebook describes the March slice I will use, checks its structure, builds five features, and shows how a leaked answer can make a quick model look much better than it is.

For access, accept the warehouse's data-use terms and store a plain Read token as `HF_TOKEN` in Colab Secrets or the local environment. The token is never written into a cell or printed. Only March is downloaded and cached; the final month stays untouched.

## 1. Unit of analysis + time window

Here are my five contract answers in plain words:

1. **What one row means:** In the source, one row is one page, for one client, on one report date. My feature frame will have one row per page within a client at a single decision date.
2. **Which table I use:** `fact_content_daily_performance`, only its `month=2026-03` partition, from the fixed warehouse release. I am using search measurements, so I do not need a content or query-table join for this exercise.
3. **Which dates I use:** The source is March 1–31, 2026. Features use March 1–14. My assumed decision date is March 18, leaving three full days after the feature window for reporting delay. The outcome uses March 18–31. Both measurement windows are 14 days long; March 15–17 is a gap. Actual reporting delays still need checking before operational use.
4. **What I predict or rank:** Rank pages for an editor's review using a short-window decline target: did impressions fall by more than 20% in the later 14 days compared with the earlier 14 days? This is a risk proxy, not evidence that a refresh would help.
5. **What I leave out:** Impressions from the outcome window are excluded from model inputs because they would not be known when the editor makes the decision. I also leave out the final-month sample and the fixed 90-day query table, whose window could overlap an outcome.

The initial review pool needs at least 100 impressions and 14 days of usable search measurements in the feature window. This is a working demand floor, not a proven cutoff. A target is filled only when the later window also has all 14 usable days; otherwise it remains missing. Later coverage is an evaluation requirement, not an input used to decide a page's priority.

In [1]:
import importlib.util
import subprocess
import sys

packages = {"duckdb": "duckdb", "huggingface_hub": "huggingface_hub",
            "sklearn": "scikit-learn", "pyarrow": "pyarrow"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Warehouse and modelling packages are available.")

Warehouse and modelling packages are available.


## 2. Fields: feature / label / context / excluded

| Role | Fields I use | Why |
|---|---|---|
| Feature | `log_impressions`, `log_clicks`, `ctr_pct`, `weighted_position`, `active_search_days` | All five use only March 1–14 search measurements. |
| Label / proxy | Later-window impression total, its coverage count, `future_decline` | These describe what happened after the decision, so they cannot be model inputs. |
| Context | `client_hash_id`, `content_hash_id`, `report_date`, availability flags, earlier-window coverage and raw totals | They identify rows, define windows, check coverage, and support grouping or calculations. IDs are never predictors. |
| Excluded | `leaked_future_decline` after the demonstration; all other warehouse fields and tables | The deliberate leak copies the target. Unused fields are outside this small contract. |

Raw inputs to the five features are `gsc_impressions`, `gsc_clicks`, and `gsc_avg_position`, restricted to the earlier window. The same impression column in the later window is a label input instead. Its time window determines its role.

I keep rows only when `gsc_data_available IS TRUE`. False and missing flags are not usable search coverage. Analytics availability is counted for comparison but not required, because these five features do not use Analytics. Search position zero is treated as unavailable, and the position feature may remain missing. Any imputation is fitted on training pages only. I will not assume an absent day means zero impressions.

In [2]:
import os
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import snapshot_download
from huggingface_hub.utils import disable_progress_bars
from IPython.display import display
disable_progress_bars()

REPO_ID = "FlyRank/internship-warehouse"
REVISION = "50cbf7c3909d07be4d1b5906b4d09e882e5acbf2"
PARTITION = "fact_content_daily_performance/month=2026-03/*.parquet"
token = os.environ.get("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
if not token:
    raise RuntimeError("Set HF_TOKEN in the environment or Colab Secrets after accepting warehouse access. Do not put the token in a cell.")

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "AGENTS.md").is_file()), Path.cwd())
cache_dir = repo_root / ".venv" / "warehouse-cache"
cache_dir.mkdir(parents=True, exist_ok=True)
try:
    snapshot = Path(snapshot_download(
        repo_id=REPO_ID, repo_type="dataset", revision=REVISION,
        allow_patterns=[PARTITION], token=token, cache_dir=str(cache_dir),
    ))
except Exception:
    raise RuntimeError("Warehouse download failed. Check gate acceptance, token permissions, and network access.") from None
finally:
    del token

files = sorted(str(p) for p in snapshot.glob(PARTITION))
assert files, "The pinned release did not provide a March partition."
con = duckdb.connect()
source = con.from_parquet(files, union_by_name=True)
required = {"report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
            "ga4_data_available", "gsc_impressions", "gsc_clicks", "gsc_avg_position"}
assert required.issubset(source.columns), "Check the release schema before running the contract."
source.create_view("march")
print("Pinned dataset revision:", REVISION)
print("Loaded only the March 2026 partition; no final-month sample or full-table scan.")

Pinned dataset revision: 50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
Loaded only the March 2026 partition; no final-month sample or full-table scan.


## 3. Verify it with queries, then build features and try the trap

I use **exactly three verification queries** below. The first checks the source grain, the second measures the March count and date span, and the third checks which rows survive the availability filter. Feature construction comes afterwards.

### Query 1: Does one source row really mean one page-day within a client?

The duplicate-key, extra-row, and missing-key counts should all be zero. The result shows aggregate counts rather than page identifiers.

In [3]:
grain_sql = """
WITH keys AS (
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_per_key
    FROM march
    GROUP BY report_date, client_hash_id, content_hash_id
)
SELECT COUNT(*) AS distinct_page_day_keys,
       COUNT(*) FILTER (WHERE rows_per_key > 1) AS duplicated_keys,
       COALESCE(SUM(rows_per_key - 1), 0) AS extra_rows,
       COUNT(*) FILTER (WHERE report_date IS NULL OR client_hash_id IS NULL
                         OR content_hash_id IS NULL) AS missing_keys
FROM keys
"""
print(grain_sql)
grain = con.sql(grain_sql).df()
display(grain)
assert grain.loc[0, ["duplicated_keys", "extra_rows", "missing_keys"]].eq(0).all()



WITH keys AS (
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS rows_per_key
    FROM march
    GROUP BY report_date, client_hash_id, content_hash_id
)
SELECT COUNT(*) AS distinct_page_day_keys,
       COUNT(*) FILTER (WHERE rows_per_key > 1) AS duplicated_keys,
       COALESCE(SUM(rows_per_key - 1), 0) AS extra_rows,
       COUNT(*) FILTER (WHERE report_date IS NULL OR client_hash_id IS NULL
                         OR content_hash_id IS NULL) AS missing_keys
FROM keys



,distinct_page_day_keys,duplicated_keys,extra_rows,missing_keys
0,9841378,0,0.0,0


### Query 2: How large is the March slice, and which dates does it cover?

These are counts for this partition, not the warehouse's full 78,835,655-row daily table. The date span should stay inside March, and coverage can still differ by page.

In [4]:
count_sql = """
SELECT COUNT(*) AS march_rows,
       MIN(report_date) AS first_date, MAX(report_date) AS last_date,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS pages
FROM march
"""
print(count_sql)
counts = con.sql(count_sql).df()
display(counts)
assert counts.loc[0, "march_rows"] == grain.loc[0, "distinct_page_day_keys"]
assert pd.Timestamp(counts.loc[0, "first_date"]) == pd.Timestamp("2026-03-01")
assert pd.Timestamp(counts.loc[0, "last_date"]) == pd.Timestamp("2026-03-31")


SELECT COUNT(*) AS march_rows,
       MIN(report_date) AS first_date, MAX(report_date) AS last_date,
       COUNT(DISTINCT client_hash_id) AS clients,
       COUNT(DISTINCT content_hash_id) AS pages
FROM march



,march_rows,first_date,last_date,clients,pages
0,9841378,2026-03-01,2026-03-31,55,331437


### Query 3: How many rows have usable tracking?

`IS TRUE` excludes both false flags and missing flags. My search-only slice uses the GSC survivor count; requiring both systems would unnecessarily remove pages from this exercise. The missing-measurement counts check whether a true flag is enough on its own.

In [5]:
availability_sql = """
SELECT COUNT(*) AS all_march_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_rows_survive,
       COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_rows_excluded,
       COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_flag_missing,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows_survive,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND ga4_data_available IS TRUE) AS both_rows_survive,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND gsc_impressions IS NULL) AS gsc_rows_missing_impressions,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND gsc_clicks IS NULL) AS gsc_rows_missing_clicks
FROM march
"""
print(availability_sql)
availability = con.sql(availability_sql).df()
display(availability)
assert availability.loc[0, "all_march_rows"] == counts.loc[0, "march_rows"]
assert (availability.loc[0, "gsc_rows_survive"] + availability.loc[0, "gsc_rows_excluded"]
        == availability.loc[0, "all_march_rows"])
print(f"Search-only rows retained: {availability.loc[0, 'gsc_rows_survive']:,}.")


SELECT COUNT(*) AS all_march_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_rows_survive,
       COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS gsc_rows_excluded,
       COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS gsc_flag_missing,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_rows_survive,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND ga4_data_available IS TRUE) AS both_rows_survive,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND gsc_impressions IS NULL) AS gsc_rows_missing_impressions,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                         AND gsc_clicks IS NULL) AS gsc_rows_missing_clicks
FROM march



,all_march_rows,gsc_rows_survive,gsc_rows_excluded,gsc_flag_missing,ga4_rows_survive,both_rows_survive,gsc_rows_missing_impressions,gsc_rows_missing_clicks
0,9841378,3611061,6230317,0,413966,364347,0,0


Search-only rows retained: 3,611,061.


**What these checks showed:** March has **9,841,378 rows**, covering **March 1–31**, **55 clients**, and **331,437 content items**. There are no duplicate page-day keys or missing keys. Requiring usable search tracking keeps **3,611,061 rows** and leaves out **6,230,317**. Requiring both search and analytics would keep only **364,347 rows**, which is why this first frame stays focused on search measurements.

The available search rows have no missing impression or click values. That does not mean every page has a full history; I check window coverage again when building the page-level frame.

### My five features

The decision is on March 18, and every feature stops at March 14. I am assuming the reporting gap makes those measurements available; a real system would need to check its reporting delay.

| Feature | Available when? |
|---|---|
| `log_impressions` | Knowable at the decision moment because it is `log1p` of impressions recorded on March 1–14. |
| `log_clicks` | Knowable at the decision moment because it uses only clicks recorded on March 1–14. |
| `ctr_pct` | Knowable at the decision moment because both its click numerator and impression denominator stop at March 14. |
| `weighted_position` | Knowable at the decision moment because it averages March 1–14 daily positions, weighted by that period's impressions; zero or missing positions are left out. |
| `active_search_days` | Knowable at the decision moment because it counts earlier-window days with positive impressions. |

CTR is calculated as clicks divided by impressions times 100. Some features share information; this is a small starting frame, not a final claim about which signals matter. IDs, coverage fields, and the target are separate from the five model inputs.

The following SQL builds page-level totals from the already cached March table. It is feature construction, separate from the three verification queries above.

In [6]:
feature_sql = """
WITH covered AS (
    SELECT *, report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' AS earlier,
              report_date BETWEEN DATE '2026-03-18' AND DATE '2026-03-31' AS later
    FROM march
    WHERE gsc_data_available IS TRUE
), totals AS (
    SELECT client_hash_id, content_hash_id,
           COUNT(*) FILTER (WHERE earlier AND gsc_impressions IS NOT NULL
                              AND gsc_clicks IS NOT NULL) AS earlier_days,
           COUNT(*) FILTER (WHERE later AND gsc_impressions IS NOT NULL) AS later_days,
           SUM(gsc_impressions) FILTER (WHERE earlier) AS earlier_impressions,
           SUM(gsc_clicks) FILTER (WHERE earlier) AS earlier_clicks,
           SUM(gsc_impressions) FILTER (WHERE later) AS later_impressions,
           COUNT(*) FILTER (WHERE earlier AND gsc_impressions > 0) AS active_search_days,
           SUM(gsc_impressions * gsc_avg_position) FILTER (
               WHERE earlier AND gsc_impressions > 0 AND gsc_avg_position > 0
                 AND isfinite(gsc_avg_position))
           / NULLIF(SUM(gsc_impressions) FILTER (
               WHERE earlier AND gsc_impressions > 0 AND gsc_avg_position > 0
                 AND isfinite(gsc_avg_position)), 0) AS weighted_position
    FROM covered
    WHERE earlier OR later
    GROUP BY client_hash_id, content_hash_id
)
SELECT * FROM totals
WHERE earlier_days = 14 AND earlier_impressions >= 100
"""
pages = con.sql(feature_sql).df()
assert not pages.empty, "No pages meet this contract; reconsider coverage before modelling."
pages["log_impressions"] = np.log1p(pages["earlier_impressions"])
pages["log_clicks"] = np.log1p(pages["earlier_clicks"])
pages["ctr_pct"] = 100 * pages["earlier_clicks"] / pages["earlier_impressions"]
pages["future_decline"] = pd.Series(pd.NA, index=pages.index, dtype="Int64")
label_available = pages["later_days"].eq(14)
pages.loc[label_available, "future_decline"] = (
    pages.loc[label_available, "later_impressions"]
    .lt(0.8 * pages.loc[label_available, "earlier_impressions"]).astype("int64")
)
FEATURES = ["log_impressions", "log_clicks", "ctr_pct", "weighted_position", "active_search_days"]
feature_frame = pages[FEATURES].copy()
assert feature_frame.shape[1] == 5
assert not pages.duplicated(["client_hash_id", "content_hash_id"]).any()
assert not np.isinf(feature_frame.to_numpy(dtype=float)).any()
print(f"Review pool: {len(pages):,} pages with 14 usable earlier days and at least 100 impressions.")
print(f"Observed later-window targets: {label_available.sum():,}; unknown targets: {(~label_available).sum():,}.")
display(pages[["client_hash_id", "content_hash_id", *FEATURES, "future_decline"]].head(5))
display(pd.DataFrame({"missing_values": feature_frame.isna().sum(),
                      "missing_pct": 100 * feature_frame.isna().mean()}).round(2))
# Check whether missing position coverage differs across clients, without displaying IDs.
position_missing_by_client = pages.groupby("client_hash_id")["weighted_position"].apply(lambda s: s.isna().mean())
print(f"Position missingness across clients: {100 * position_missing_by_client.min():.2f}% to "
      f"{100 * position_missing_by_client.max():.2f}%.")

Review pool: 60,534 pages with 14 usable earlier days and at least 100 impressions.
Observed later-window targets: 57,459; unknown targets: 3,075.


,client_hash_id,content_hash_id,log_impressions,log_clicks,ctr_pct,weighted_position,active_search_days,future_decline
0,client_73cda7b4e4f265ea,content_f96cfaec4b8b7782,8.491055,2.079442,0.143737,1.765708,14,0
1,client_73cda7b4e4f265ea,content_b313e40c45d42243,7.175490,2.079442,0.535988,6.415008,14,1
2,client_73cda7b4e4f265ea,content_37ef521cd8beae2b,5.834811,1.098612,0.586510,9.736070,14,0
3,client_73cda7b4e4f265ea,content_bf2c1278879f0de7,6.304449,0.693147,0.183150,1.664835,14,1
4,client_73cda7b4e4f265ea,content_e7ad90637b7fd582,8.975504,4.127134,0.771566,3.826461,14,0


,missing_values,missing_pct
log_impressions,0,0.0
log_clicks,0,0.0
ctr_pct,0,0.0
weighted_position,0,0.0
active_search_days,0,0.0


Position missingness across clients: 0.00% to 0.00%.


### The deliberate leak

I use a small decision tree with one fixed client-held-out split. It is a quick lesson, not a tuned model or a capstone result. The clean version gets only the five earlier-window features. The second version gets one extra column that is literally a copy of the later-window label. That column would not exist at the decision moment.

I compare ROC-AUC on exactly the same test pages, so the only difference is the leaked input. ROC-AUC checks whether declined pages tend to get higher scores than other pages: 0.5 is chance ordering and 1.0 is perfect ordering. I also show precision@20 within test clients, with shorter lists reported separately. A perfect ranking of labels still would not prove an editor should refresh those pages.

In [7]:
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

experiment = pages.loc[pages["future_decline"].notna()].reset_index(drop=True)
X = experiment[FEATURES].copy()
y = experiment["future_decline"].astype(int)
groups = experiment["client_hash_id"]
assert groups.nunique() >= 4 and y.nunique() == 2, "Need enough clients and both target classes."
train_index, test_index = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42).split(X, y, groups))
assert set(groups.iloc[train_index]).isdisjoint(set(groups.iloc[test_index]))
assert y.iloc[train_index].nunique() == y.iloc[test_index].nunique() == 2

def quick_model():
    return make_pipeline(SimpleImputer(strategy="median", keep_empty_features=True),
                         DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=42))

clean_model = quick_model()
clean_model.fit(X.iloc[train_index], y.iloc[train_index])
clean_probability = clean_model.predict_proba(X.iloc[test_index])[:, 1]
clean_auc = roc_auc_score(y.iloc[test_index], clean_probability)

LEAK = "leaked_future_decline"
leaked_X = X.assign(**{LEAK: y})
leaked_model = quick_model()
leaked_model.fit(leaked_X.iloc[train_index], y.iloc[train_index])
leaked_probability = leaked_model.predict_proba(leaked_X.iloc[test_index])[:, 1]
leaked_auc = roc_auc_score(y.iloc[test_index], leaked_probability)

print(f"Train: {len(train_index):,} pages across {groups.iloc[train_index].nunique()} clients.")
print(f"Test: {len(test_index):,} pages across {groups.iloc[test_index].nunique()} different clients.")
display(pd.DataFrame({"version": ["Five earlier-window features", "Same five plus a copy of the answer"],
                      "ROC_AUC": [clean_auc, leaked_auc]}).round(4))
print(f"Score change after adding the leaked answer: {leaked_auc - clean_auc:+.4f} ROC-AUC.")

test_pages = experiment.iloc[test_index][["client_hash_id", "content_hash_id", "future_decline"]].copy()
test_pages["clean_score"] = clean_probability
top20 = (test_pages.sort_values(["client_hash_id", "clean_score", "content_hash_id"],
                               ascending=[True, False, True])
         .groupby("client_hash_id", sort=False).head(20))
per_client = top20.groupby("client_hash_id")["future_decline"].agg(["mean", "size"])
full = per_client.loc[per_client["size"].eq(20)]
short = per_client.loc[per_client["size"].lt(20)]
if not full.empty:
    print(f"Clean mean precision@20: {100 * full['mean'].mean():.2f}% across {len(full)} full lists.")
print(f"Shorter test-client lists: {len(short)}.")
if not short.empty:
    print(f"Their mean decline share: {100 * short['mean'].mean():.2f}%.")

Train: 14,070 pages across 24 clients.
Test: 43,389 pages across 8 different clients.


,version,ROC_AUC
0,Five earlier-window features,0.5975
1,Same five plus a copy of the answer,1.0000


Score change after adding the leaked answer: +0.4025 ROC-AUC.
Clean mean precision@20: 49.29% across 7 full lists.
Shorter test-client lists: 1.
Their mean decline share: 33.33%.


**What happened:** The clean model scored **0.5975 ROC-AUC**. Adding one copy of the future label pushed that to **1.0000** on the same test pages. The perfect score came from giving the model the answer, not from discovering a useful pattern. I will keep the clean result and treat it as a limited starting comparison, not a finished review system.

### Remove the leak and keep the clean result

The extra column was an answer, not a useful signal. I remove it and the leaked model below, and rerun the clean model with the same split. The retained feature frame contains exactly the original five features. The comparison above stays visible as evidence of the mistake, but the leaked model is not kept or exported.

In [8]:
leaked_X.drop(columns=[LEAK], inplace=True)
assert list(leaked_X.columns) == FEATURES
del leaked_X, leaked_model, leaked_probability
final_features = X.copy()
assert final_features.shape[1] == 5 and LEAK not in final_features.columns
final_model = quick_model()
final_model.fit(final_features.iloc[train_index], y.iloc[train_index])
final_auc = roc_auc_score(y.iloc[test_index], final_model.predict_proba(final_features.iloc[test_index])[:, 1])
assert np.isclose(final_auc, clean_auc)
print(f"Retained clean ROC-AUC: {final_auc:.4f}.")
print("Leaked column and model removed; five earlier-window features remain.")

Retained clean ROC-AUC: 0.5975.
Leaked column and model removed; five earlier-window features remain.


## 4. Data limits

**My main limitation is the short time window.** This is one mid-panel month and two 14-day comparisons. It does not test seasonality, long-term behaviour, or the 30-day target I described in Week 2. That longer target still needs a separate contract and more history.

The experiment only evaluates pages with complete search coverage in both windows. Pages that started tracking late or lost coverage can be left out, so the results may not represent the whole inventory. A true availability flag also does not guarantee that the page has useful search volume. I report the review pool and missing targets rather than silently replacing missing history with zero.

Client holdout stops a client's pages appearing on both sides of this split, but one month's check is still not a test across future seasons. The split holds out about a quarter of the clients, not a quarter of the pages: the eight test clients contain 43,389 pages, while the 24 training clients contain 14,070. The pooled ROC-AUC can therefore be influenced by larger clients. The per-client precision@20 is a separate view, and neither result is a claim that ML beats a simple review rule.

I cannot tell whether editing a page would prevent decline: there are no interventions or editor judgments here. The clean result is a limited observed comparison, and the leaked result is only a demonstration of why future information must stay out of the inputs.

In [9]:
print(f"Source window: {pd.Timestamp(counts.loc[0, 'first_date']).date()} to "
      f"{pd.Timestamp(counts.loc[0, 'last_date']).date()}.")
print("Features: March 1–14; assumed decision: March 18; outcome: March 18–31.")
print(f"Evaluation coverage: {len(experiment):,} / {len(pages):,} earlier-eligible pages have later labels.")
print(f"Observed decline rate among those labelled pages: {100 * y.mean():.2f}%.")
print("No causal refresh claim; no final-month data used; no raw data exported into git.")

Source window: 2026-03-01 to 2026-03-31.
Features: March 1–14; assumed decision: March 18; outcome: March 18–31.
Evaluation coverage: 57,459 / 60,534 earlier-eligible pages have later labels.
Observed decline rate among those labelled pages: 36.64%.
No causal refresh claim; no final-month data used; no raw data exported into git.


## 5. Self-check

- [x] I answered the five contract questions in plain words.
- [x] The three verification queries ran, with their outputs saved and availability checked using `IS TRUE`.
- [x] I built exactly five features and explained why each is knowable at the decision moment.
- [x] I compared the clean score with one deliberate leaked column on the same held-out clients.
- [x] I removed the leaked column and model and kept the clean result.
- [x] I named the short-window and coverage limitations.
- [x] The notebook runs top to bottom with saved outputs.
- [x] Public output is limited to aggregate results and pseudonymized examples, with no credentials or private names, URLs, or queries.
- [x] The executed notebook is committed under `work/notebooks/`.

My repository: [FlyrankAI-Internship](https://github.com/vibhanshu-s/FlyrankAI-Internship).